# AIRF SLM tool calling — FunctionGemma 270M + synthetic data (T4, 1 epoch)

Gemma/FunctionGemma on 15k only plateaued at **~88.9%** dev.
This run mixes **official 15k + 9k synthetic** (new names, places, templates) so the model learns rules, not train sentences.
FunctionGemma 270M, **1 epoch**, T4.

Licence: https://huggingface.co/google/functiongemma-270m-it


In [ ]:
import importlib.util, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao", "peft", "accelerate", "transformers", "huggingface_hub"])
print("ok")


In [ ]:
import json, math, os, random, re, time, urllib.request, zipfile, gc
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("device", DEVICE, DTYPE)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("colab userdata:", e)
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if HF_TOKEN:
    HF_TOKEN = str(HF_TOKEN).strip().strip('"').strip("'")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN introuvable. Colab → 🔑 Secrets → HF_TOKEN + interrupteur Notebook access ON. "
        "Runtime → Restart session."
    )
if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError("Le secret HF_TOKEN ne ressemble pas à un token Hugging Face (doit commencer par hf_).")

from huggingface_hub import login, whoami, HfApi
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

info = whoami(token=HF_TOKEN)
print("HF account:", info.get("name"), "| type:", info.get("auth", {}).get("type") or info.get("type"))
print("Si 403 Gemma: ouvre https://huggingface.co/google/gemma-3-270m-it CONNECTÉ EN TANT QUE", info.get("name"))
print("puis clique Agree and access. Attends 1–2 min, Runtime → Restart, relance.")

api = HfApi(token=HF_TOKEN)
for repo in ("google/functiongemma-270m-it", "google/gemma-3-270m-it"):
    try:
        api.model_info(repo)
        print("ACCESS OK", repo)
    except Exception as e:
        print("NO ACCESS", repo, "->", type(e).__name__, str(e)[:300])
        if repo == "google/functiongemma-270m-it":
            raise RuntimeError(
                "Accepte la licence FunctionGemma: https://huggingface.co/google/functiongemma-270m-it "
                "(même compte que HF_TOKEN), puis Runtime → Restart."
            ) from e


In [ ]:
REPO = "https://raw.githubusercontent.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon/main/data"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
for name in ("tools.json", "train.jsonl", "dev.jsonl"):
    urllib.request.urlretrieve(f"{REPO}/{name}", DATA_DIR / name)

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def answer_of(row):
    return row["tool_calls"] if "tool_calls" in row else {"tool": row["tool"], "parameters": row["parameters"]}

def calls_of(answer):
    return answer if isinstance(answer, list) else [answer]

tools = {t["name"]: t for t in json.loads((DATA_DIR / "tools.json").read_text())}
train = [dict(row, answer=answer_of(row)) for row in read_jsonl(DATA_DIR / "train.jsonl")]
dev = read_jsonl(DATA_DIR / "dev.jsonl")
print(len(tools), "tools", len(train), "train", len(dev), "dev")

SYN_URL = "https://raw.githubusercontent.com/maick-code/airf-slm-tool-calling-hackathon/arena/7d628867-airf-slm-tool-calling-hackatho/data/synthetic.jsonl"
try:
    urllib.request.urlretrieve(SYN_URL, DATA_DIR / "synthetic.jsonl")
    syn = []
    for row in read_jsonl(DATA_DIR / "synthetic.jsonl"):
        ans = row.get("answer") or (row["tool_calls"] if "tool_calls" in row else {"tool": row["tool"], "parameters": row["parameters"]})
        syn.append(dict(row, answer=ans))
    train = train + syn
    print("synthetic", len(syn), "total train", len(train))
except Exception as e:
    print("synthetic not loaded:", e)


In [ ]:
# T4: FunctionGemma 270M, 1 epoch (less overfit than Gemma×2).
MODEL_NAME = "google/functiongemma-270m-it"
EPOCHS = 1
LORA_R = 16
LR = 2e-4
BATCH_TOKENS = 2048
INSTRUCTION = (
    "Reply with a JSON list of the tool calls for the request, with null for values not given.\nTools:\n"
)

print("loading tokenizer", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(example):
    signatures = [f"{name}({', '.join(f'{p}: {t}' for p, t in tools[name]['parameters'].items())})" for name in example["tools"]]
    messages = [{"role": "system", "content": INSTRUCTION + "\n".join(signatures)}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def target_text(answer):
    return json.dumps(calls_of(answer), ensure_ascii=False) + "<end_of_turn>"

by_category = defaultdict(list)
for ex in train:
    by_category[ex["meta"]["category"]].append(ex)
validation = [ex for group in by_category.values() for ex in random.Random(SEED).sample(group, 100)]
held = {ex["id"] for ex in validation}
train_pool = [ex for ex in train if ex["id"] not in held]
print(MODEL_NAME, "epochs", EPOCHS, "train_pool", len(train_pool), "val", len(validation))
print("cuda mem allocated GB", round(torch.cuda.memory_allocated()/1e9, 2) if torch.cuda.is_available() else 0)


In [ ]:
def normalise(value):
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    if isinstance(value, str):
        return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", value.lower())).strip()
    return value

def canonical(answer):
    try:
        return tuple((c["tool"], tuple(sorted((k, normalise(v)) for k, v in c["parameters"].items() if v is not None))) for c in calls_of(answer))
    except (TypeError, KeyError, AttributeError):
        return None

def score(predictions):
    hits = defaultdict(list)
    for ex in validation:
        hits[ex["meta"]["category"]].append(canonical(predictions.get(ex["id"])) == canonical(ex["answer"]))
    return {"overall": float(np.mean(sum(hits.values(), [])))} | {k: float(np.mean(hits[k])) for k in ("single", "missing_info", "multiple")}

def parse(text):
    text = text.split("<end_of_turn>")[0]
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    try:
        value = json.JSONDecoder().raw_decode(text, min(starts))[0] if starts else None
    except json.JSONDecodeError:
        return None
    return value[0] if isinstance(value, list) and len(value) == 1 else value

@torch.no_grad()
def predict(model, examples, batch_size=8):
    model.eval()
    order = sorted(examples, key=lambda ex: len(build_prompt(ex)))
    stop = [tokenizer.eos_token_id]
    eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if isinstance(eot, int) and eot >= 0:
        stop.append(eot)
    out = {}
    for i in range(0, len(order), batch_size):
        batch = order[i:i+batch_size]
        inputs = tokenizer([build_prompt(ex) for ex in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        tokens = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=stop, pad_token_id=tokenizer.pad_token_id)
        for ex, seq in zip(batch, tokens[:, inputs["input_ids"].shape[1]:]):
            out[ex["id"]] = parse(tokenizer.decode(seq, skip_special_tokens=False))
        if i % 128 == 0:
            print("pred", i, "/", len(order))
    return out


In [ ]:
import gc

def free_gpu():
    for obj_name in ("model", "base"):
        if obj_name in globals():
            try:
                del globals()[obj_name]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("cuda free GB", round(torch.cuda.mem_get_info()[0]/1e9, 2))

def fine_tune(examples, epochs=2, lr=LR, batch_tokens=BATCH_TOKENS, lora_r=LORA_R):
    """Starter-style LoRA on Gemma 3 270M — fits a Colab T4 (15GB)."""
    free_gpu()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
        attn_implementation="eager",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    lora_config = LoraConfig(task_type="CAUSAL_LM", r=lora_r, lora_alpha=32, target_modules="all-linear")
    model = get_peft_model(model, lora_config)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()

    rows = []
    for ex in examples:
        prompt = tokenizer(build_prompt(ex), add_special_tokens=False)["input_ids"]
        answer = tokenizer(target_text(ex["answer"]), add_special_tokens=False)["input_ids"]
        rows.append((prompt + answer, [-100] * len(prompt) + answer))

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    start = time.time()
    base = model.get_base_model()
    for ep in range(epochs):
        random.Random(SEED + ep).shuffle(rows)
        packed = sorted(rows, key=lambda r: len(r[0]) // 32)
        batches, batch = [], []
        for row in packed:
            if batch and (len(batch) + 1) * len(row[0]) > batch_tokens:
                batches.append(batch)
                batch = []
            batch.append(row)
        if batch:
            batches.append(batch)
        random.Random(SEED + ep).shuffle(batches)
        nsteps = max(1, len(batches))
        schedule = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda s, n=nsteps: min(1, (s + 1) / 10) * 0.5 * (1 + math.cos(math.pi * s / n))
        )
        model.train()
        for bi, batch in enumerate(batches):
            width = max(len(ids) for ids, _ in batch)
            ids = torch.tensor([[tokenizer.pad_token_id] * (width - len(i)) + i for i, _ in batch], device=DEVICE)
            labels = torch.tensor([[-100] * (width - len(l)) + l for _, l in batch], device=DEVICE)
            mask = (ids != tokenizer.pad_token_id).long()
            hidden = base.model(
                input_ids=ids,
                attention_mask=mask,
                position_ids=(mask.cumsum(1) - 1).clamp(min=0),
            ).last_hidden_state
            targets, keep = labels[:, 1:], labels[:, 1:] != -100
            logits = base.lm_head(hidden[:, :-1][keep]).float()
            loss = torch.nn.functional.cross_entropy(logits, targets[keep])
            if math.isfinite(loss.item()):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            schedule.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
            del ids, labels, mask, hidden, logits, loss
            if bi % 50 == 0:
                print(f"epoch {ep+1}/{epochs} step {bi}/{len(batches)} last={losses[-1]:.4f} alloc={torch.cuda.memory_allocated()/1e9:.2f}GB")
                torch.cuda.empty_cache()
    print(f"done {len(examples)} x {epochs} ep in {(time.time()-start)/60:.1f} min")
    return model, losses


In [ ]:
# After OOM: Runtime → Disconnect and delete runtime, then Run all.
free_gpu()
model, losses = fine_tune(train_pool, epochs=EPOCHS)
val_score = score(predict(model, validation, batch_size=8))
print("validation", val_score)
print("Dev leaderboard last run was 88.9% with Gemma×2. FunctionGemma×1 should generalize better.")


In [ ]:
adapter_dir = Path("adapter")
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved", adapter_dir.resolve())

predictions = predict(model, dev)
empty = {"tool": "", "parameters": {}}
payload = {k: (v if v is not None else empty) for k, v in predictions.items()}
Path("submission.json").write_text(json.dumps(payload, ensure_ascii=False, indent=1))
Path("predictions.json").write_text(json.dumps(payload, ensure_ascii=False))
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("predictions.json")
    z.write("submission.json")
print(len(payload), "-> submission.zip  (upload this on Codabench)")


In [ ]:
# Push submission.zip to GitHub. Secret names accepted: GH_TOKEN or GITHUB_TOKEN.
# Notebook access must be ON for that secret (same as HF_TOKEN).
import os, shutil, subprocess, tempfile
from pathlib import Path

GH_REPO = os.environ.get("GH_REPO", "https://github.com/maick-code/airf-slm-tool-calling-hackathon.git")
BRANCH = os.environ.get("GH_BRANCH", "arena/7d628867-airf-slm-tool-calling-hackatho")

def read_gh_token():
    names = ("GH_TOKEN", "GITHUB_TOKEN", "github_token")
    errors = []
    try:
        from google.colab import userdata
        for name in names:
            try:
                val = userdata.get(name)
                if val:
                    return str(val).strip().strip('"').strip("'"), name
            except Exception as e:
                errors.append(f"{name}: {type(e).__name__}: {e}")
    except Exception as e:
        errors.append(f"userdata: {e}")
    for name in names:
        val = os.environ.get(name)
        if val:
            return val.strip(), name
    print("GitHub token not found. Tried", names)
    print("Errors:", errors or "(none — secret missing or Notebook access OFF)")
    return None, None

gh_token, gh_name = read_gh_token()
if not gh_token:
    print("No GitHub token. Colab → 🔑 Secrets → add GH_TOKEN (or GITHUB_TOKEN), enable Notebook access, Runtime → Restart, rerun THIS cell only.")
    print("Meanwhile: Files panel → download submission.zip → upload to Codabench.")
else:
    print("using secret", gh_name, "len", len(gh_token))
    if not Path("submission.zip").exists():
        raise FileNotFoundError("submission.zip missing — run the previous cell first")
    auth = GH_REPO.replace("https://", f"https://x-access-token:{gh_token}@")
    work = Path(tempfile.mkdtemp(prefix="airf-push-"))
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--single-branch", "--depth", "1", auth, str(work)])
    for name in ("submission.json", "submission.zip", "predictions.json"):
        if Path(name).exists():
            shutil.copy2(name, work / name)
    subprocess.check_call(["git", "config", "user.email", "colab@local"], cwd=work)
    subprocess.check_call(["git", "config", "user.name", "colab"], cwd=work)
    subprocess.check_call(["git", "add", "-A"], cwd=work)
    rc = subprocess.call(["git", "commit", "-m", "Colab Gemma 270M submission"], cwd=work)
    subprocess.check_call(["git", "push", "origin", BRANCH], cwd=work)
    print("pushed to", BRANCH, "(commit skipped)" if rc else "")
